In [1]:
import numpy as np
import gudhi as gd
import pandas as pd
import networkx as nx

# Function to load the distance matrix from an Excel file
def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)  # Capture up to 2D simplices
    return simplex_tree

# Function to compute persistent homology and track hole formation
def compute_persistence_1D(simplex_tree):
    persistence = simplex_tree.persistence()
    H1_intervals = []
    hole_data = []
    
    # Sort edges by filtration value
    edges = []
    faces = []
    for simplex, filtration_value in simplex_tree.get_filtration():
        if len(simplex) == 2:  # Edges
            edges.append((simplex[0], simplex[1], filtration_value))
        elif len(simplex) == 3:  # Triangles (2-simplices)
            faces.append((simplex[0], simplex[1], simplex[2], filtration_value))

    edges.sort(key=lambda x: x[2])  # Sort edges by filtration value (birth time)

    # Dynamic graph construction to track cycle formation
    G = nx.Graph()
    for i, interval in enumerate(persistence):
        if interval[0] == 1:  # H1 homology (1D holes)
            birth, death = interval[1]
            H1_intervals.append((birth, death))

            # Add edges progressively until reaching birth time
            G.clear()
            for u, v, filt in edges:
                if filt > birth:
                    break
                G.add_edge(u, v)

            # Detect the first cycle that forms
            cycles = list(nx.cycle_basis(G))
            participating_vertices = set()
            for cycle in cycles:
                if len(cycle) > 2:  # Ensure it's a valid hole (not a trivial triangle)
                    participating_vertices.update(cycle)

            hole_data.append({
                "hole_index": i + 1,
                "birth": birth,
                "death": death,
                "vertices": sorted(participating_vertices)
            })

    return H1_intervals, hole_data

# Main part of the script to compute and record protein involvement in holes

# Load the distance matrix
distance_matrix_file = "trial_distance_matrix_500.xlsx"  # Change this to your file path
distance_matrix = load_distance_matrix(distance_matrix_file)

# Set the max edge length for the Rips complex
max_edge_length = np.max(distance_matrix)

# Compute persistence intervals and hole data
H1_intervals, hole_data = compute_persistence_1D(compute_rips_from_distance_matrix(distance_matrix, max_edge_length))

# Initialize a dictionary to store protein hole participation
protein_holes = {i+1: {"births": [], "deaths": []} for i in range(distance_matrix.shape[0])}

# Record the birth and death times for each protein
for hole in hole_data:
    hole_index = hole["hole_index"]
    birth = hole["birth"]
    death = hole["death"]
    vertices = hole["vertices"]
    
    # For each protein in the hole, record its birth and death times
    for protein in vertices:
        protein_holes[protein+1]["births"].append((hole_index, birth))
        protein_holes[protein+1]["deaths"].append((hole_index, death))

# Prepare the data for export to Excel
protein_holes_for_export = []
for protein, holes in protein_holes.items():
    for hole_index, birth in holes["births"]:
        death = next((d for idx, d in holes["deaths"] if idx == hole_index), "infinity")
        protein_holes_for_export.append([protein, hole_index, birth, death])

# Create a DataFrame to hold the results
columns = ['Protein', 'Hole Index', 'Birth', 'Death']
protein_holes_df = pd.DataFrame(protein_holes_for_export, columns=columns)

# Export the results to an Excel file
output_file = "protein_holes_results_h1_try.xlsx"  # Change this to your desired output file path
protein_holes_df.to_excel(output_file, index=False)

# Print message confirming the export
print(f"Protein hole data has been exported to {output_file}")


Protein hole data has been exported to protein_holes_results_h1_try.xlsx


In [2]:
import numpy as np
import pandas as pd

# Function to compute the persistence landscape for a single feature with birth and death times
def persistence_landscape(b, d, X):
    """
    Computes the persistence landscape for a single feature with birth time 'b' and death time 'd'.
    
    Parameters:
    - b: Birth time of the feature.
    - d: Death time of the feature.
    - X: Array of evaluation points.
    
    Returns:
    - Array of landscape values at each point in X.
    """
    landscape_values = []
    for i in X:
        # Compute min(i - b, d - i) for the given birth and death times
        value = min(i - b, d - i)
        landscape_values.append(value)
    
    return np.array(landscape_values)

# Load the protein hole data from the Excel file
excel_file = "protein_holes_results_h1_try.xlsx"  # Change to your file path
df = pd.read_excel(excel_file)

# X: Points at which we evaluate the landscape (for example, from 0 to 4)
X = np.linspace(0, 4, 100)

# Function to compute the L1, L2, Linf norms of the persistence landscape
def compute_norms(landscape):
    L1_norm = np.sum(np.abs(landscape))  # L1 norm
    L2_norm = np.sqrt(np.sum(landscape ** 2))  # L2 norm
    Linf_norm = np.max(np.abs(landscape))  # Linf norm
    return L1_norm, L2_norm, Linf_norm

# Prepare a list to store results
results = []

# Group the data by protein and calculate the persistence landscape norms
for protein in df['Protein'].unique():
    # Get the rows corresponding to the current protein
    protein_data = df[df['Protein'] == protein]
    
    # Initialize arrays to accumulate the landscape values
    total_landscape = np.zeros(len(X))
    
    for _, row in protein_data.iterrows():
        birth = row['Birth']
        death = row['Death']
        
        # Compute the persistence landscape for this hole
        landscape = persistence_landscape(birth, death, X)
        total_landscape += landscape  # Accumulate the landscape values
    
    # Compute the norms for this protein
    L1_norm, L2_norm, Linf_norm = compute_norms(total_landscape)
    
    # Append the result for this protein
    results.append([protein, L1_norm, L2_norm, Linf_norm])

# Create a DataFrame to store the results
norms_df = pd.DataFrame(results, columns=['Protein', 'L1 Norm', 'L2 Norm', 'Linf Norm'])

# Export the results to an Excel file
output_file = "protein_norms_h1.xlsx"  # Change to your desired output file path
norms_df.to_excel(output_file, index=False)

# Print message confirming the export
print(f"Protein norms have been exported to {output_file}")


Protein norms have been exported to protein_norms_h1.xlsx


In [ ]:
import numpy as np
import gudhi as gd
import pandas as pd
import networkx as nx

# Function to load the distance matrix from an Excel file
def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)  # Capture up to 2D simplices
    return simplex_tree

# Function to compute persistent homology and track hole formation
def compute_persistence_1D(simplex_tree):
    persistence = simplex_tree.persistence()
    H1_intervals = []
    hole_data = []

    # Sort edges by filtration value
    edges = []
    faces = []
    for simplex, filtration_value in simplex_tree.get_filtration():
        if len(simplex) == 2:  # Edges
            edges.append((simplex[0], simplex[1], filtration_value))
        elif len(simplex) == 3:  # Triangles (2-simplices)
            faces.append((simplex[0], simplex[1], simplex[2], filtration_value))

    edges.sort(key=lambda x: x[2])  # Sort edges by filtration value (birth time)

    # Dynamic graph construction to track cycle formation
    G = nx.Graph()
    for i, interval in enumerate(persistence):
        if interval[0] == 1:  # H1 homology (1D holes)
            birth, death = interval[1]
            H1_intervals.append((birth, death))

            # Add edges progressively until reaching birth time
            G.clear()
            for u, v, filt in edges:
                if filt > birth:
                    break
                G.add_edge(u, v)

            # Detect the first cycle that forms
            cycles = list(nx.cycle_basis(G))
            participating_vertices = set()
            for cycle in cycles:
                if len(cycle) > 2:  # Ensure it's a valid hole (not a trivial triangle)
                    participating_vertices.update(cycle)

            hole_data.append({
                "hole_index": i + 1,
                "birth": birth,
                "death": death,
                "vertices": sorted(participating_vertices)
            })

    return H1_intervals, hole_data

# Main part of the script to compute and save results to Excel

# Load the distance matrix
distance_matrix_file = "trial_distance_matrix_500.xlsx"  # Change this to your file path
distance_matrix = load_distance_matrix(distance_matrix_file)

# Set the max edge length for the Rips complex
max_edge_length = np.max(distance_matrix)

# Compute persistence intervals and hole data
H1_intervals, hole_data = compute_persistence_1D(compute_rips_from_distance_matrix(distance_matrix, max_edge_length))

# Prepare the data for export to Excel
hole_data_for_export = []
for hole in hole_data:
    birth = hole["birth"]
    death = hole["death"] if hole["death"] != float("inf") else "infinity"
    vertices = ", ".join([f"Protein {v+1}" for v in hole["vertices"]])
    hole_data_for_export.append([hole["hole_index"], birth, death, vertices])

# Create a DataFrame to hold the results
columns = ['Hole Index', 'Birth', 'Death', 'Vertices']
hole_data_df = pd.DataFrame(hole_data_for_export, columns=columns)

# Export the results to an Excel file
output_file = "persistence_holes_results_h1.xlsx"  # Change this to your desired output file path
hole_data_df.to_excel(output_file, index=False)

# Print message confirming the export
print(f"Results have been exported to {output_file}")
